## Tool (工具调用)

```
LangChain Tool（新版 1.0）
├── 定义工具：@tool 装饰器（最推荐）
│     ├── name（函数名）
│     ├── description（docstring）
│     └── args_schema（类型注解 → Pydantic）
├── 组装 Agent：create_agent(model, tools=[...], system_prompt, middleware)
├── 运行：agent.invoke({"messages": [...]})
├── 核心机制：模型输出 tool_call → 框架执行函数 → 结果回传（ReAct 循环）
└── 1.0 新变化：统一 create_agent、弃用 ToolNode、Middleware 中间件、wrap_tool_call 错误处理
```

### 自定义工具案例

In [5]:
# 1. 必须知道各种工具的使用说明
# 2. 当遇到具体场景，能够从工具库中选择合适工具使用
# ============ Step 1：导入 ============
from langchain.agents import create_agent            # 新版统一入口
from langchain_core.tools import tool                # @tool 装饰器
from langchain.chat_models import init_chat_model    # ChatModel

# ============ Step 2：定义工具 ============
# 注意：函数的描述必须写在函数体中的第一行
@tool
def get_weather(city: str) -> str:
    """查询指定城市的实时天气。
    参数 city: 城市英文名，例如 Beijing、Shanghai。
    """
    # 真实场景这里应该调天气 API，这里用假数据演示
    return f"{city} 今天晴，25°C"

import datetime
@tool
def get_date():
    """ 获取今天的具体日期 """
    return datetime.date.today().strftime("%Y-%m-%d")

import webbrowser
@tool
def open_browser(url: str) -> str:
    """打开浏览器访问指定网址"""
    try:
        webbrowser.open(url)
        return f"已打开浏览器访问 {url}"
    except Exception as e:
        return f"打开浏览器失败: {str(e)}"


# ============ Step 3：创建模型 ============
model = init_chat_model(model="deepseek-flash")

# ============ Step 4：创建 Agent（把工具交给模型）============
from langgraph.checkpoint.memory import InMemorySaver
memory = InMemorySaver() # 创建短期记忆实例：智能体中使用InMemorySaver()实现单会话的短期记忆
agent = create_agent(
    model=model,
    tools=[get_weather, get_date, open_browser],          # 关键：这里传工具列表
    system_prompt= "你是人工智能助手。需要帮助用户解决各种问题。", # 【可选】系统提示词
    checkpointer=memory, # 【可选】记忆对象
)

# ============ Step 5：运行 ============
result = agent.invoke(
    {"messages": [{"role": "user", "content": "北京今天天气怎么样？"}]},
    config={"configurable": {"thread_id": "user_1"}}  # 会话唯一标识，用于区分不同用户
)
# result = agent.invoke({"messages":[{"role":"user","content":"今天是几月几号？"}]})
# result = agent.invoke({"messages":[{"role":"user","content":"帮我打开淘宝"}]})

# ============ Step 6：取最终回答 ============
print(result["messages"][-1].content)

北京今天的天气情况如下：

- **日期**：2026年10月10日
- **天气**：晴 ☀️
- **气温**：25°C

今天北京天气晴朗，气温舒适宜人，非常适合外出活动。不过秋季昼夜温差较大，建议您早晚适当增添衣物，注意保暖。☺️


In [2]:
print(type(result))
print(type(result['messages']))

<class 'dict'>
<class 'list'>


### 前端交互

In [2]:
# 与前端交互处理LLM响应
def process_llm_response(query, history):
    # type="messages" 模式下，history 是 [{"role":..., "content":...}, ...] 格式的列表
    history = history or []
    if len(query) == 0:
        return history
    user_msg = {"role": "user", "content": query}
    try:
        # 先展示用户消息 + "正在查询"占位
        yield history + [user_msg, {"role": "assistant", "content": "正在查询大模型..."}], ""

        # 使用Agent处理查询
        response = agent.invoke(
            {"messages": [{"role": "user", "content": query}]},
            # 配置会话标识，用于区分不同用户
            config={"configurable": {"thread_id": "user_1"}}  # 会话唯一标识，用于区分不同用户
        )
        print(f"LLM输出：{response}")

        # 正确提取回答内容 - 只显示output部分
        if isinstance(response, dict) and 'messages' in response:
            answer = response["messages"][-1].content
        else:
            # 如果response不是预期的字典格式，尝试提取其他可能的字段
            answer = str(response)
            print(f"警告: 响应格式异常: {response}")

        # 返回结果
        yield history + [user_msg, {"role": "assistant", "content": answer}], ""
    except Exception as e:
        print(f"Error: {e}")
        yield history + [user_msg, {"role": "assistant", "content": "AI助手出错，请重试或者检查"}], ""


import gradio as gr
# 前端界面展示
with gr.Blocks(title="大模型中Function Call演示") as demo:
    gr.HTML('<center><h1>欢迎大模型中Function Call演示</h1></center>')

    with gr.Row():
        with gr.Column(scale=10):
            chatbot = gr.Chatbot(height=650, type="messages")  # 空初始值

    with gr.Row():
        msg = gr.Textbox(label="输入", placeholder="您想了解什么呢？")

    with gr.Row():
        examples = gr.Examples(
            examples=[
                '请问如何做红烧牛肉？',
                '料酒可以换成白酒吗？',
                '帮我打开计算器',
                '现在几点了？',
                '帮我访问淘宝网',
                '今天昆山天气怎么样？'
            ],
            inputs=[msg]
        )

    clear = gr.ClearButton([chatbot, msg])
    msg.submit(process_llm_response, [msg, chatbot], [chatbot,msg])

# 下面把本地回环地址加入 NO_PROXY，让自检请求绕过代理。
import os
os.environ["NO_PROXY"] = "localhost,127.0.0.1"
os.environ["no_proxy"] = "localhost,127.0.0.1"
if __name__ == '__main__':
    demo.launch(server_port=7778)

E:\Agent\L2阶段\langchain_tutorial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7778
* To create a public link, set `share=True` in `launch()`.
